In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [22]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [23]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [24]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
      

In [25]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John,\n\nNo problem—thanks for letting me know. We can reschedule. What time tomorrow works for you? I'm flexible and can adjust to your availability. If it helps, I can propose a couple of options, but I’m happy to hear what you prefer.\n\nBest,\nSeán"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John,\\n\\nNo problem—thanks for letting me know. We can reschedule. What time tomorrow works for you? I\'m flexible and can adjust to your availability. If it helps, I can propose a couple of options, but I’m happy to hear what you prefer.\\n\\nBest,\\nSeán"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='a871ea73697555502791669fb1af73c5')]


In [26]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for letting me know. We can reschedule. What time tomorrow works for you? I'm flexible and can adjust to your availability. If it helps, I can propose a couple of options, but I’m happy to hear what you prefer.

Best,
Seán


## Approve

In [8]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1e34442a-3cf6-4298-85e1-3db7222e44f3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 531, 'prompt_tokens': 167, 'total_tokens': 698, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EHg7dLWhNavYm7xop5sy8fnE5fiuu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04615-5fd3-70e0-a1cc

## Reject

In [ ]:
# If this cell is run after the approve cell, it will resume the conversation from the last approved tool call, so reject won't work. Need to run the agent invoke cell again to test the flow.
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Of '
                                                                          'course—we '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'What '
                                                                          'time '
                                                                          'would '
                                                                          'work '
                                                                          'best '
         

In [19]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

Of course—we can reschedule. What time would work best for you? I’m available tomorrow afternoon after 2:00 PM or Thursday morning before 11:00 AM. If those don’t fit, please suggest a time and I’ll adjust.

Your merciful leader,
Seán


## Edit

In [27]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          "Let's "
                                                                          'reschedule '
                                                                          'for '
                                                                          'tomorrow. '
                                                                          'Wh

In [28]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem—thanks for the heads-up. Let's reschedule for tomorrow. What times work for you? I can do 10:00 AM, 2:00 PM, or 4:00 PM. If none of these fit, please suggest a couple of alternatives and I’ll adjust. Best regards, Seán
